This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch transformers --upgrade -q

## Language models and the Transformer

### The language model

#### Training a Shakespeare language model

In [ ]:
# PyTorch: keras.utils.get_file becomes a plain urllib download.
import urllib.request

url = (
    "https://storage.googleapis.com/download.tensorflow.org/"
    "data/shakespeare.txt"
)
filename, _ = urllib.request.urlretrieve(url)
shakespeare = open(filename, "r").read()

In [0]:
shakespeare[:250]

In [ ]:
# PyTorch: no tf.data.Dataset; we keep the features/labels as plain Python
# lists of equal-length character strings and batch them ourselves later.
sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))

In [ ]:
x, y = features[0], labels[0]
x[:50], y[:50]

In [ ]:
# PyTorch: there is no layers.TextVectorization. We build the equivalent
# character-level vocabulary ourselves. Index 0 is reserved for padding ("")
# and index 1 for [UNK], matching Keras' default reserved tokens.
chars = sorted(set(shakespeare))
vocabulary = ["", "[UNK]"] + chars
char_to_id = {c: i for i, c in enumerate(vocabulary)}

def tokenizer(text, sequence_length=sequence_length):
    ids = [char_to_id.get(c, 1) for c in text][:sequence_length]
    ids = ids + [0] * (sequence_length - len(ids))
    return ids

In [ ]:
vocabulary_size = len(vocabulary)
vocabulary_size

In [ ]:
# PyTorch: map the tokenizer over every example and wrap the integer ids in a
# TensorDataset + shuffled DataLoader (replaces dataset.map/.shuffle/.batch).
import torch
from torch.utils.data import TensorDataset, DataLoader

feature_ids = torch.tensor([tokenizer(t) for t in features], dtype=torch.long)
label_ids = torch.tensor([tokenizer(t) for t in labels], dtype=torch.long)
training_dataset = TensorDataset(feature_ids, label_ids)
training_data = DataLoader(training_dataset, batch_size=64, shuffle=True)

In [ ]:
from torch import nn

embedding_dim = 256
hidden_dim = 1024

# PyTorch: the functional Keras model becomes an nn.Module. We drop the final
# softmax because nn.CrossEntropyLoss expects raw logits. nn.GRU is batch_first
# and returns (sequence_output, last_state); we keep the full sequence here.
class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.dense = nn.Linear(hidden_dim, vocabulary_size)

    def forward(self, inputs, initial_state=None):
        x = self.embedding(inputs)
        x, state = self.gru(x, initial_state)
        x = self.dropout(x)
        return self.dense(x), state

model = LanguageModel()

In [ ]:
# PyTorch: print(module) is the closest equivalent to Keras' model.summary().
print(model)

In [ ]:
# PyTorch: compile()+fit() become an explicit optimizer/loss and training loop.
# CrossEntropyLoss expects (N, C) logits and (N,) targets, so we flatten the
# time dimension into the batch.
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(20):
    model.train()
    for inputs, targets in training_data:
        logits, _ = model(inputs)
        loss = loss_fn(logits.reshape(-1, vocabulary_size), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}: loss {loss.item():.4f}")

#### Generating Shakespeare

In [ ]:
# PyTorch: we don't need a separate generation model. The same LanguageModel
# already returns its GRU state, so we can feed one token at a time and thread
# the state through. (Keras built a second model only because its functional GRU
# layer didn't expose the state in the training graph.)
generation_model = model

In [ ]:
token_ids = range(vocabulary_size)
id_to_char = dict(zip(token_ids, vocabulary))

prompt = """
KING RICHARD III:
"""

In [ ]:
# PyTorch: run the prompt through the model one token at a time under no_grad to
# build up the GRU hidden state (replaces generation_model.predict in a loop).
input_ids = [char_to_id[c] for c in prompt]
state = torch.zeros(1, 1, hidden_dim)
generation_model.eval()
with torch.no_grad():
    for token_id in input_ids:
        inputs = torch.tensor([[token_id]])
        logits, state = generation_model(inputs, state)
        predictions = logits[:, -1, :]

In [ ]:
generated_ids = []
max_length = 250
with torch.no_grad():
    for i in range(max_length):
        next_char = int(predictions.argmax(dim=-1)[0])
        generated_ids.append(next_char)
        inputs = torch.tensor([[next_char]])
        logits, state = generation_model(inputs, state)
        predictions = logits[:, -1, :]

In [0]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)

### Sequence-to-sequence learning

#### English-to-Spanish translation

In [ ]:
# PyTorch: download + extract the spa-eng archive with the standard library
# instead of keras.utils.get_file.
import pathlib
import urllib.request
import zipfile

url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
zip_file, _ = urllib.request.urlretrieve(url)
extract_dir = pathlib.Path("spa-eng-data")
with zipfile.ZipFile(zip_file) as z:
    z.extractall(extract_dir)
text_path = extract_dir / "spa-eng" / "spa.txt"

In [0]:
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

In [0]:
import random
random.choice(text_pairs)

In [0]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [ ]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

# PyTorch: replace layers.TextVectorization with a small word-level vectorizer.
# Standardization (lowercase + strip punctuation) and whitespace splitting match
# Keras' defaults; the Spanish side keeps "[start]"/"[end]" markers. Reserved
# ids: 0 = padding, 1 = [UNK], matching Keras.
def custom_standardization(input_string):
    lowercase = input_string.lower()
    return re.sub(f"[{re.escape(strip_chars)}]", "", lowercase)

def default_standardization(input_string):
    lowercase = input_string.lower()
    return re.sub(f"[{re.escape(string.punctuation)}]", "", lowercase)

class TextVectorizer:
    def __init__(self, max_tokens, output_sequence_length, standardize):
        self.max_tokens = max_tokens
        self.output_sequence_length = output_sequence_length
        self.standardize = standardize
        self.vocabulary = None
        self.token_to_id = None

    def adapt(self, texts):
        from collections import Counter
        counts = Counter()
        for text in texts:
            counts.update(self.standardize(text).split())
        most_common = [w for w, _ in counts.most_common(self.max_tokens - 2)]
        self.vocabulary = ["", "[UNK]"] + most_common
        self.token_to_id = {w: i for i, w in enumerate(self.vocabulary)}

    def get_vocabulary(self):
        return self.vocabulary

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        out = []
        L = self.output_sequence_length
        for text in texts:
            ids = [self.token_to_id.get(w, 1) for w in self.standardize(text).split()][:L]
            ids = ids + [0] * (L - len(ids))
            out.append(ids)
        return torch.tensor(out, dtype=torch.long)

vocab_size = 15000
sequence_length = 20

english_tokenizer = TextVectorizer(
    max_tokens=vocab_size,
    output_sequence_length=sequence_length,
    standardize=default_standardization,
)
spanish_tokenizer = TextVectorizer(
    max_tokens=vocab_size,
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

In [ ]:
# PyTorch: build tensors once with the tokenizers, then serve dict-style batches
# from a DataLoader. The decoder input is spa[:, :-1], the target is spa[:, 1:],
# and sample_weights mask out padding (id 0) just like the Keras pipeline.
batch_size = 64

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng = english_tokenizer(list(eng_texts))
    spa = spanish_tokenizer(list(spa_texts))
    decoder_input = spa[:, :-1]
    labels = spa[:, 1:]
    dataset = TensorDataset(eng, decoder_input, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

def batch_to_inputs(eng, spa, labels):
    features = {"english": eng, "spanish": spa}
    sample_weights = (labels != 0).float()
    return features, labels, sample_weights

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [ ]:
eng, spa, labels = next(iter(train_ds))
inputs, targets, sample_weights = batch_to_inputs(eng, spa, labels)
print(inputs["english"].shape)

In [ ]:
print(inputs["spanish"].shape)

In [ ]:
print(targets.shape)

In [ ]:
print(sample_weights.shape)

#### Sequence-to-sequence learning with RNNs

In [ ]:
embed_dim = 256
hidden_dim = 1024

# PyTorch: the encoder half of the functional seq2seq model. A bidirectional GRU
# with merge_mode="sum" => we add the two direction final states. mask_zero has
# no direct nn equivalent; padding is handled by the masked loss instead.
class Seq2SeqEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True,
                          bidirectional=True)

    def forward(self, source):
        x = self.embedding(source)
        _, state = self.gru(x)          # state: (2, batch, hidden_dim)
        return state[0] + state[1]      # merge_mode="sum" over directions

In [ ]:
# PyTorch: the decoder half + the full model. The encoder's merged state seeds
# the decoder GRU's initial hidden state; we drop the softmax (CrossEntropyLoss
# wants logits).
class Seq2SeqRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Seq2SeqEncoder()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.dense = nn.Linear(hidden_dim, vocab_size)

    def forward(self, source, target):
        encoder_output = self.encoder(source)
        x = self.embedding(target)
        x, _ = self.gru(x, encoder_output.unsqueeze(0))
        x = self.dropout(x)
        return self.dense(x)

seq2seq_rnn = Seq2SeqRNN()

In [ ]:
print(seq2seq_rnn)

In [ ]:
# PyTorch: compile()/fit() -> training loop. We mask padding tokens (id 0) by
# passing ignore_index=0 to CrossEntropyLoss, which mirrors the sample_weights
# masking + weighted accuracy used on the Keras side.
optimizer = torch.optim.Adam(seq2seq_rnn.parameters())
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

def run_epoch(model, loader, train):
    model.train(train)
    total_loss = total_correct = total_tokens = 0.0
    torch.set_grad_enabled(train)
    for eng, spa, labels in loader:
        logits = model(eng, spa)
        loss = loss_fn(logits.reshape(-1, vocab_size), labels.reshape(-1))
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        mask = labels != 0
        preds = logits.argmax(dim=-1)
        total_correct += ((preds == labels) & mask).sum().item()
        total_tokens += mask.sum().item()
        total_loss += loss.item() * mask.sum().item()
    torch.set_grad_enabled(True)
    return total_loss / total_tokens, total_correct / total_tokens

for epoch in range(15):
    tr_loss, tr_acc = run_epoch(seq2seq_rnn, train_ds, train=True)
    val_loss, val_acc = run_epoch(seq2seq_rnn, val_ds, train=False)
    print(f"Epoch {epoch}: loss {tr_loss:.4f} acc {tr_acc:.4f} "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    seq2seq_rnn.eval()
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        with torch.no_grad():
            next_token_predictions = seq2seq_rnn(
                tokenized_input_sentence, tokenized_target_sentence
            ).numpy()
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

### The Transformer architecture

#### Dot-product attention

#### Transformer encoder block

In [ ]:
# PyTorch: the custom Keras layer becomes an nn.Module. layers.MultiHeadAttention
# -> nn.MultiheadAttention(batch_first=True). The boolean source_mask (True =
# real token) is inverted into a key_padding_mask (True = position to ignore).
# Post-norm residual structure is preserved exactly.
class TransformerEncoder(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )
        self.self_attention_layernorm = nn.LayerNorm(hidden_dim)
        self.feed_forward_1 = nn.Linear(hidden_dim, intermediate_dim)
        self.feed_forward_2 = nn.Linear(intermediate_dim, hidden_dim)
        self.feed_forward_layernorm = nn.LayerNorm(hidden_dim)

    def forward(self, source, source_mask):
        residual = x = source
        key_padding_mask = ~source_mask
        x, _ = self.self_attention(
            query=x, key=x, value=x, key_padding_mask=key_padding_mask
        )
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = torch.relu(self.feed_forward_1(x))
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Transformer decoder block

In [ ]:
# PyTorch: the decoder layer. use_causal_mask=True -> we build a causal
# attn_mask for the self-attention. Cross-attention uses the encoder padding mask
# as key_padding_mask. Same post-norm residual structure as the book.
class TransformerDecoder(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )
        self.self_attention_layernorm = nn.LayerNorm(hidden_dim)
        self.cross_attention = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )
        self.cross_attention_layernorm = nn.LayerNorm(hidden_dim)
        self.feed_forward_1 = nn.Linear(hidden_dim, intermediate_dim)
        self.feed_forward_2 = nn.Linear(intermediate_dim, hidden_dim)
        self.feed_forward_layernorm = nn.LayerNorm(hidden_dim)

    def forward(self, target, source, source_mask):
        residual = x = target
        seq_len = x.shape[1]
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1
        )
        x, _ = self.self_attention(
            query=x, key=x, value=x, attn_mask=causal_mask
        )
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x, _ = self.cross_attention(
            query=x, key=source, value=source, key_padding_mask=~source_mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = torch.relu(self.feed_forward_1(x))
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Sequence-to-sequence learning with a Transformer

In [ ]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

# PyTorch: the functional encoder-decoder Transformer becomes an nn.Module. The
# source padding mask (source != 0) is computed inside forward and threaded into
# both the encoder and the decoder cross-attention. Final softmax dropped.
class Transformer(nn.Module):
    def __init__(self, embedding_factory):
        super().__init__()
        self.source_embedding = embedding_factory()
        self.target_embedding = embedding_factory()
        self.encoder = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)
        self.decoder = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)
        self.dropout = nn.Dropout(0.5)
        self.dense = nn.Linear(hidden_dim, vocab_size)

    def forward(self, source, target):
        source_mask = source != 0
        x = self.source_embedding(source)
        encoder_output = self.encoder(source=x, source_mask=source_mask)
        x = self.target_embedding(target)
        x = self.decoder(target=x, source=encoder_output, source_mask=source_mask)
        x = self.dropout(x)
        return self.dense(x)

transformer = Transformer(lambda: nn.Embedding(vocab_size, hidden_dim))

In [ ]:
print(transformer)

In [ ]:
# PyTorch: same masked training loop as the RNN model (reusing run_epoch).
optimizer = torch.optim.Adam(transformer.parameters())
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(15):
    tr_loss, tr_acc = run_epoch(transformer, train_ds, train=True)
    val_loss, val_acc = run_epoch(transformer, val_ds, train=False)
    print(f"Epoch {epoch}: loss {tr_loss:.4f} acc {tr_acc:.4f} "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

#### Embedding positional information

In [ ]:
# PyTorch: keras.ops.* -> torch.*; positions come from arange. Token + position
# embeddings are summed, same as the book.
class PositionalEmbedding(nn.Module):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = nn.Embedding(input_dim, output_dim)
        self.position_embeddings = nn.Embedding(sequence_length, output_dim)

    def forward(self, inputs):
        seq_len = inputs.shape[-1]
        positions = torch.arange(seq_len, device=inputs.device)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

In [ ]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

# PyTorch: rebuild the Transformer, this time using PositionalEmbedding for both
# the source and target embeddings (same Transformer nn.Module as before).
transformer = Transformer(
    lambda: PositionalEmbedding(sequence_length + 1, vocab_size, hidden_dim)
)

In [ ]:
optimizer = torch.optim.Adam(transformer.parameters())
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(30):
    tr_loss, tr_acc = run_epoch(transformer, train_ds, train=True)
    val_loss, val_acc = run_epoch(transformer, val_ds, train=False)
    print(f"Epoch {epoch}: loss {tr_loss:.4f} acc {tr_acc:.4f} "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    transformer.eval()
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        with torch.no_grad():
            next_token_predictions = transformer(
                tokenized_input_sentence, tokenized_target_sentence
            ).numpy()
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

### Classification with a pretrained Transformer

#### Pretraining a Transformer encoder

#### Loading a pretrained Transformer

In [ ]:
# PyTorch: keras_hub presets -> HuggingFace transformers. The roberta_base_en
# preset maps to the "roberta-base" checkpoint. The backbone (no task head) is
# AutoModel; the tokenizer is AutoTokenizer.
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
backbone = AutoModel.from_pretrained("roberta-base")

In [ ]:
# PyTorch: HuggingFace tokenizers return a dict of input_ids / attention_mask.
tokenizer("The quick brown fox")

In [ ]:
print(backbone)

#### Preprocessing IMDb movie reviews

In [ ]:
import os, pathlib, shutil, random
import urllib.request
import tarfile

# PyTorch: download + extract the IMDb archive with the standard library.
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
tar_file, _ = urllib.request.urlretrieve(url)
imdb_root = pathlib.Path("imdb_data")
with tarfile.open(tar_file) as t:
    t.extractall(imdb_root)

imdb_extract_dir = imdb_root / "aclImdb"
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(train_dir / category, exist_ok=True)
    os.makedirs(val_dir / category, exist_ok=True)
    for index, file in enumerate(src_files):
        if index < num_val_samples:
            shutil.copy(src_dir / file, val_dir / category / file)
        else:
            shutil.copy(src_dir / file, train_dir / category / file)

In [ ]:
# PyTorch: text_dataset_from_directory has no direct equivalent. We read the .txt
# files from each class folder into (text, label) lists; label 0 = neg, 1 = pos
# follows Keras' alphabetical class ordering.
def load_text_directory(directory):
    directory = pathlib.Path(directory)
    texts, labels = [], []
    for label, category in enumerate(("neg", "pos")):
        for file in sorted((directory / category).glob("*.txt")):
            texts.append(file.read_text(encoding="utf-8"))
            labels.append(label)
    return texts, labels

batch_size = 16
train_texts, train_labels = load_text_directory(train_dir)
val_texts, val_labels = load_text_directory(val_dir)
test_texts, test_labels = load_text_directory(test_dir)

In [ ]:
# PyTorch: keras_hub.layers.StartEndPacker becomes the HF tokenizer's built-in
# truncation/padding (it already adds the <s>/</s> tokens and an attention_mask).
# We build DataLoaders that yield {token_ids, padding_mask} dicts + labels.
def preprocess(texts, labels):
    encoded = tokenizer(
        texts,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    inputs = {
        "token_ids": encoded["input_ids"],
        "padding_mask": encoded["attention_mask"],
    }
    return inputs, torch.tensor(labels, dtype=torch.float32)

def make_text_dataset(texts, labels, shuffle):
    inputs, y = preprocess(texts, labels)
    dataset = TensorDataset(inputs["token_ids"], inputs["padding_mask"], y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

preprocessed_train_ds = make_text_dataset(train_texts, train_labels, shuffle=True)
preprocessed_val_ds = make_text_dataset(val_texts, val_labels, shuffle=False)
preprocessed_test_ds = make_text_dataset(test_texts, test_labels, shuffle=False)

In [ ]:
token_ids, padding_mask, label = next(iter(preprocessed_train_ds))
{"token_ids": token_ids, "padding_mask": padding_mask}, label

#### Fine-tuning a pretrained Transformer

In [ ]:
# PyTorch: build the classification head on top of the backbone. x[:, 0, :]
# takes the [CLS]/<s> token. We drop the final sigmoid and train with
# BCEWithLogitsLoss (numerically stable, expects logits).
class Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.dropout_1 = nn.Dropout(0.1)
        self.dense = nn.Linear(backbone.config.hidden_size, 768)
        self.dropout_2 = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 1)

    def forward(self, token_ids, padding_mask):
        x = self.backbone(input_ids=token_ids, attention_mask=padding_mask)
        x = x.last_hidden_state[:, 0, :]
        x = self.dropout_1(x)
        x = torch.relu(self.dense(x))
        x = self.dropout_2(x)
        return self.classifier(x).squeeze(-1)

classifier = Classifier(backbone)

In [ ]:
# PyTorch: fine-tuning loop. Adam(5e-5), BCEWithLogitsLoss for the binary task.
optimizer = torch.optim.Adam(classifier.parameters(), lr=5e-5)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(1):
    classifier.train()
    for token_ids, padding_mask, labels in preprocessed_train_ds:
        logits = classifier(token_ids, padding_mask)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    classifier.eval()
    val_correct = val_total = 0
    with torch.no_grad():
        for token_ids, padding_mask, labels in preprocessed_val_ds:
            logits = classifier(token_ids, padding_mask)
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.numel()
    print(f"Epoch {epoch}: val_acc {val_correct / val_total:.4f}")

In [ ]:
# PyTorch: classifier.evaluate() -> a manual no-grad accuracy/loss pass.
classifier.eval()
test_correct = test_total = 0
total_loss = 0.0
with torch.no_grad():
    for token_ids, padding_mask, labels in preprocessed_test_ds:
        logits = classifier(token_ids, padding_mask)
        total_loss += loss_fn(logits, labels).item() * labels.numel()
        preds = (torch.sigmoid(logits) > 0.5).float()
        test_correct += (preds == labels).sum().item()
        test_total += labels.numel()
print(f"loss: {total_loss / test_total:.4f} - accuracy: {test_correct / test_total:.4f}")

### What makes the Transformer effective?